# Interactive Gate Exploration (Website Layer)

Explore how the shared governance engine reaches an APPROVE/REJECT verdict under fixed, pre-defined scenarios.

## How to use this notebook
> Run all cells, then use the controls below.

## Interactive Controls
Select a scenario and profile, and use the checkboxes to reveal more details.

## Live Outputs
Below the controls you will see (1) a gates chart, (2) a verdict panel, and (3) a plain-language interpretation panel.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import ipywidgets as widgets
from IPython.display import clear_output, display

# === NON-CANONICAL WEBSITE INTERACTIVE NOTEBOOK ===
# Presentation layer only: all calculations route to the authoritative engine.

def repo_root() -> Path:
    """Resolve repository root from env override or parent-anchor scan."""
    env_root = os.environ.get("EAA_REPO_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "engine" / "corrected_public_engine_v1_1.py").is_file():
            return p

    start = Path.cwd().resolve()
    anchors = [
        Path("engine") / "corrected_public_engine_v1_1.py",
        Path("config") / "notebook_plan.json",
        Path("reproduce_all.py"),
    ]
    for p in (start, *start.parents):
        if all((p / a).exists() for a in anchors):
            return p
    raise RuntimeError(
        f"Repository root not found from cwd={start}. Expected anchors: {anchors}"
    )


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "engine"))
import corrected_public_engine_v1_1 as eng

# Pre-defined scenarios only (same feature physics the archival notebooks use).
SCENARIOS = [
    {
        "label": "smoke_core_001 (replay reference)",
        "case": {
            "case_id": "smoke_core_001",
            "features": {
                "intrinsic_safety": 0.58,
                "evidence_strength": 0.55,
                "bias_harm_index": 0.44,
                "uncertainty_calibration": 0.52,
                "traceability_integrity": 0.53,
            },
        },
        "mode": eng.MODE_REPLAY,
    },
    {
        "label": "demo_replay_pass (full mode)",
        "case": {
            "case_id": "demo_replay_pass",
            "features": {
                "intrinsic_safety": 0.62,
                "evidence_strength": 0.60,
                "bias_harm_index": 0.40,
                "uncertainty_calibration": 0.58,
                "traceability_integrity": 0.57,
            },
        },
        "mode": eng.MODE_CANONICAL_FULL,
    },
    {
        "label": "demo_gate_fail (full mode)",
        "case": {
            "case_id": "demo_gate_fail",
            "features": {
                "intrinsic_safety": 0.35,
                "evidence_strength": 0.60,
                "bias_harm_index": 0.40,
                "uncertainty_calibration": 0.58,
                "traceability_integrity": 0.57,
            },
        },
        "mode": eng.MODE_CANONICAL_FULL,
    },
    {
        "label": "demo_fullmode_abstention",
        "case": {
            "case_id": "demo_fullmode_abstention",
            "features": {
                "intrinsic_safety": 0.62,
                "evidence_strength": 0.60,
                "bias_harm_index": 0.40,
                "uncertainty_calibration": 0.42,
                "traceability_integrity": 0.50,
                "fallback_safety_delta": 0.15,
            },
        },
        "mode": eng.MODE_CANONICAL_FULL,
    },
]

# Bounded, non-destructive exploration controls (display only)
scenario_dd = widgets.Dropdown(
    options=[s["label"] for s in SCENARIOS],
    value=SCENARIOS[0]["label"],
    description="Scenario:",
    style={"description_width": "initial"},
)

profile_dd = widgets.Dropdown(
    options=["permissive", "moderate", "strict", "very_strict"],
    value="moderate",
    description="Profile:",
    style={"description_width": "initial"},
)

components_toggle = widgets.Checkbox(
    value=True,
    description="Show gate/component values",
)

details_toggle = widgets.Checkbox(
    value=True,
    description="Show SCM/abstention details (full mode)",
)

show_chart_toggle = widgets.Checkbox(
    value=True,
    description="Show chart",
)

show_summary_toggle = widgets.Checkbox(
    value=True,
    description="Show verdict & interpretation",
)

controls_box = widgets.VBox(
    [
        scenario_dd,
        profile_dd,
        components_toggle,
        details_toggle,
        show_chart_toggle,
        show_summary_toggle,
    ]
)

chart_out = widgets.Output(
    layout={"border": "1px solid #ccc", "padding": "6px"}
)

verdict_out = widgets.Output(
    layout={"border": "1px solid #ccc", "padding": "6px"}
)

interp_out = widgets.Output(
    layout={"border": "1px solid #ccc", "padding": "6px"}
)


def render(
    selected_label: str,
    profile_name: str,
    show_components: bool,
    show_scm_details: bool,
    show_chart: bool,
    show_summary: bool,
) -> None:
    scenario = next(s for s in SCENARIOS if s["label"] == selected_label)
    result = eng.evaluate_case(
        scenario["case"], profile_name=profile_name, mode=scenario["mode"]
    )

    gate_cols = [
        ("gate_safety", "Safety"),
        ("gate_evidence", "Evidence"),
        ("gate_bias", "Bias cap"),
        ("gate_calibration", "Calibration"),
        ("gate_traceability", "Traceability"),
    ]
    gate_values = [result[k] for k, _ in gate_cols]

    with chart_out:
        clear_output(wait=True)
        if show_chart:
            plt.close("all")
            fig, ax = plt.subplots(figsize=(7, 3.5))
            ax.bar([lbl for _, lbl in gate_cols], gate_values, color="#2c6e49")
            ax.set_ylim(0, 1.05)
            ax.set_ylabel("Pass (1) / Fail (0)")
            ax.set_title(f"{scenario['label']} — {profile_name}")
            plt.tight_layout()
            plt.show()
        else:
            print("Chart hidden.")

    with verdict_out:
        clear_output(wait=True)
        if show_summary:
            summary = pd.Series(
                {
                    "governance_outcome": result["governance_outcome"],
                    "approved": result["approved"],
                    "compensatory_score": result["compensatory_score"],
                    "mode": result["mode"],
                }
            )
            print("Verdict summary (display only):")
            display(summary)

            if show_components:
                print("\nGate/component values:")
                print(f"All gates pass: {result['all_gates_pass']}")
                for key, label in gate_cols:
                    print(f"  {label}: {result[key]}")
                print(
                    f"Failed gates (binding constraints): {result.get('binding_constraints', [])}"
                )
                print(f"Compensatory approved: {result['compensatory_approved']}")
        else:
            print("Verdict summary hidden.")

    with interp_out:
        clear_output(wait=True)
        if show_summary:
            mode = result.get("mode")
            approved = int(result.get("approved"))
            if mode == eng.MODE_REPLAY:
                print("Interpretation: The bar chart shows gate pass/fail (1=pass, 0=fail) on the selected profile. The bias gate uses an inverted rule: lower bias-harm index is better, so 'Bias cap' passes at or below its threshold. In replay mode, SCM abstention/fallback is not applied; approval depends only on whether all gates pass.")
            else:
                print("Interpretation: The bar chart shows gate pass/fail (1=pass, 0=fail) on the selected profile. In full mode, the engine evaluates an SCM-derived abstention from uncertainty_calibration; if abstention triggers and fallback adequacy is not met, approval is rejected even when gates pass.")

            if approved == 1:
                print("Result: APPROVAL.")
            else:
                print("Result: REJECTION.")

            if mode == eng.MODE_CANONICAL_FULL and show_scm_details:
                if "abstention_rate" in result:
                    abst_trig = int(result.get("abstention_triggered", 0))
                    fallback_ok = int(result.get("fallback_adequate", 0))
                    print(
                        f"SCM details: abstention_rate={result.get('abstention_rate')}, "
                        f"abstention_triggered={abst_trig}, fallback_adequate={fallback_ok}."
                    )
                else:
                    print("SCM details are not available for this selected mode.")

            # Plain-language explanation for WHY, derived from the same engine fields.
            if approved == 1:
                print(
                    "Why: The selected scenario meets the conditions required for approval under the chosen mode/profile."
                )
            else:
                if result.get("all_gates_pass") == 0:
                    print("Why: One or more governance gates failed under the chosen profile.")
                elif mode == eng.MODE_CANONICAL_FULL:
                    print("Why: Gates passed, but full-mode veto conditions prevented approval.")
        else:
            print("Interpretation hidden.")

def _rerender(change=None):
    render(
        scenario_dd.value,
        profile_dd.value,
        components_toggle.value,
        details_toggle.value,
        show_chart_toggle.value,
        show_summary_toggle.value,
    )

for w in (
    scenario_dd,
    profile_dd,
    components_toggle,
    details_toggle,
    show_chart_toggle,
    show_summary_toggle,
):
    w.observe(_rerender, names="value")

# Stable display order: controls first, then outputs, then first render.
ui_layout = widgets.VBox([controls_box, chart_out, verdict_out, interp_out])
display(ui_layout)

_rerender()

